# Simulation 2 Start-of-Film Context Reinstatement Data

> Generate the Simulation 2 start-context reinstatement CSV inputs for the current script-rendered figure.

This notebook reruns the high-interference endpoint from the Simulation 2 anchor figure with periodic test-phase film cues. It compares unguided recall with start-of-film context reinstatement while target monitoring is off, then writes the row-level CSVs consumed by the script renderer.

In [ ]:
# --- figure configuration ---
PROJECT_ROOT = ""
FIT_PATH = "results/fits/Dupertuys2026_eCMR_source_only_phi_no_shared_support_best_of_1.json"
FIGURE_DIR = "figures/manuscript/simulation2"
DATA_DIR = "figure_data/manuscript/simulation2"
FIGURE_STR = "simulation2_start_context_reinstatement"
RNG_SEED = 0

N_FILM = 16
N_BREAK = 16
N_INTERFERENCE = 16
N_FILLER = 16
EXPERIMENT_COUNT = 5000
MAX_RECALL = 48

TASK_MCF_SCALE = 2.0
TASK_CONDITION_LABEL = "Strong task encoding"
REMINDER_CONDITION = "With reminder"
REMINDER_CONDITION_LABEL = "Film reminder"

FILM_CUE_REINSTATEMENT = 0.90
CUE_INTERVAL = 4
FIRST_CUE_AFTER = 0
PRIMARY_SOURCE_ORIENTATION = 0.0
SELECTED_START_DRIFT = 1.5

FILM_EMOTIONAL = True
INTERFERENCE_EMOTIONAL = False
CACHE_AFTER = "filler"

EXTRA_FIXED_SCALES = {
    "source_learning_baseline": 0.05,
    "primacy_scale": 0.1,
    "primacy_decay": 1.5,
}

BASE_FIXED_SCALES = {
    "break_drift_scale": 1.0,
    "break_mcf_scale": 1.0,
    "filler_drift_scale": 1.0,
    "filler_mcf_scale": 1.0,
}

REMINDER_SCALES = {
    "reminder_start_drift_scale": 1.0,
    "reminder_drift_scale": 1.0,
}

RETRIEVAL_FIXED_SCALES = {
    "tau_scale": 1.0,
    "target_recall_drift_scale": 1.0,
    "recall_drift_scale": 1.0,
    "rejected_recall_drift_scale": 1.0,
}

RETRIEVAL_SETTINGS = [
    {
        "label": "Unguided recall",
        "figure_label": "Unguided recall",
        "start_drift_scale": 0.0,
    },
    {
        "label": "Start-of-film context reinstatement",
        "figure_label": "Start-of-film context reinstatement",
        "start_drift_scale": SELECTED_START_DRIFT,
    },
]


In [ ]:
import csv
import os
import warnings
from pathlib import Path

import numpy as np
from IPython.display import Markdown
from jax import random
from jaxcmr.analyses.pnr import fixed_pres_pnr
from jaxcmr.analyses.spc import fixed_pres_spc
from jaxcmr.helpers import find_project_root

from selective_interference_v2 import (
    Paradigm,
    apply_supplied_item_context_cue,
    batch_trial,
    configure_rates,
    load_fit_params,
    make_factory,
    make_is_emotional,
    make_is_target,
    prepare_sweep,
    simulate_periodic_cued_free_recall,
    split_scales_for_cache,
    standard_remap,
    sweep_rngs,
)

warnings.filterwarnings("ignore")


In [ ]:
#| output: asis
if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT)
else:
    project_root = Path(find_project_root())

fit_path = project_root / FIT_PATH
figure_dir = project_root / FIGURE_DIR if FIGURE_DIR else None
data_dir = project_root / DATA_DIR if DATA_DIR else None
base_fixed_scales = {**BASE_FIXED_SCALES, **EXTRA_FIXED_SCALES}
high_interference_scales = {
    **base_fixed_scales,
    **REMINDER_SCALES,
    "interference_mcf_scale": TASK_MCF_SCALE,
}

if FIRST_CUE_AFTER != 0:
    raise ValueError("This diagnostic assumes FIRST_CUE_AFTER = 0 to match the current Simulation 2 anchor figure.")

params, n_subjects = load_fit_params(fit_path)
paradigm = Paradigm(
    n_film=N_FILM,
    n_break=N_BREAK,
    n_interference=N_INTERFERENCE,
    n_filler=N_FILLER,
    experiment_count=EXPERIMENT_COUNT,
    max_recall=MAX_RECALL,
)
factory = make_factory(
    is_emotional=make_is_emotional(
        paradigm,
        film_emotional=FILM_EMOTIONAL,
        interference_emotional=INTERFERENCE_EMOTIONAL,
    ),
    is_target=make_is_target(paradigm),
)
rng = random.PRNGKey(RNG_SEED)

Markdown(
    f"Loaded {n_subjects} fit parameter set(s). Running {len(RETRIEVAL_SETTINGS)} retrieval settings "
    f"under film reminder + strong task encoding with a film cue before first recall and "
    f"{EXPERIMENT_COUNT} simulated trials per subject."
)


In [ ]:
def position_phase_labels(paradigm):
    return (
        ["film"] * paradigm.n_film
        + ["break"] * paradigm.n_break
        + ["task"] * paradigm.n_interference
        + ["filler"] * paradigm.n_filler
    )


def write_rows(path, fieldnames, rows):
    os.makedirs(path.parent, exist_ok=True)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def phase_recall_totals(spc):
    totals = {}
    for phase in sorted(set(position_phase)):
        positions = [idx for idx, value in enumerate(position_phase) if value == phase]
        totals[phase] = float(np.sum(spc[positions]))
    return totals


def retrieval_start_support_trial(model, rng, film_items, cue_scale):
    _, cue_rng = random.split(rng)
    model = model.start_retrieving()
    cue_item = random.choice(cue_rng, film_items)
    model = apply_supplied_item_context_cue(model, cue_item, cue_scale)
    support = model.mcf.probe(model.context.state)[:n_presented]
    return support


def cued_recall_trial(
    model,
    rng,
    film_items,
    cue_scale,
    max_recall,
    cue_interval,
    first_cue_after,
):
    recall_rng, initial_cue_rng = random.split(rng)
    model = model.start_retrieving()
    if first_cue_after == 0:
        cue_item = random.choice(initial_cue_rng, film_items)
        model = apply_supplied_item_context_cue(model, cue_item, cue_scale)
        scheduled_first_cue_after = cue_interval
    else:
        scheduled_first_cue_after = first_cue_after
    _, recalls, cue_attempts, cue_items = simulate_periodic_cued_free_recall(
        model,
        max_recall,
        recall_rng,
        film_items,
        cue_scale,
        cue_interval=cue_interval,
        first_cue_after=scheduled_first_cue_after,
    )
    return recalls, cue_attempts, cue_items


position_phase = position_phase_labels(paradigm)
n_presented = len(position_phase)
batched_start_support = batch_trial(retrieval_start_support_trial, n_args=4)
batched_cued_recall = batch_trial(cued_recall_trial, n_args=7, n_static=3)


In [ ]:
pre_cache_scales, post_cache_scales = split_scales_for_cache(
    high_interference_scales,
    CACHE_AFTER,
)
prepared = prepare_sweep(
    params,
    paradigm,
    factory,
    cache_after=CACHE_AFTER,
    **pre_cache_scales,
)


In [ ]:
support_rows = []
pfr_rows = []
spc_rows = []
phase_rows = []
support_curves = {}
pfr_curves = {}
spc_curves = {}
recall_sequences = {}
run_rng = random.PRNGKey(RNG_SEED + 20)

common_metadata = {
    "reminder_condition": REMINDER_CONDITION,
    "task_condition": TASK_CONDITION_LABEL,
    "task_mcf_scale": float(TASK_MCF_SCALE),
    "film_cue_reinstatement": float(FILM_CUE_REINSTATEMENT),
    "first_cue_after": int(FIRST_CUE_AFTER),
    "cue_interval": int(CUE_INTERVAL),
    "film_source_start_drift_rate": float(PRIMARY_SOURCE_ORIENTATION),
    "rejected_recall_drift_scale": float(RETRIEVAL_FIXED_SCALES["rejected_recall_drift_scale"]),
}

for setting in RETRIEVAL_SETTINGS:
    start_drift_scale = float(setting["start_drift_scale"])
    ready_models = configure_rates(
        prepared.models,
        **post_cache_scales,
        **RETRIEVAL_FIXED_SCALES,
        start_drift_scale=start_drift_scale,
        film_source_start_drift_rate=float(PRIMARY_SOURCE_ORIENTATION),
    )
    rngs, run_rng = sweep_rngs(
        run_rng,
        prepared.n_subjects,
        prepared.experiment_count,
    )
    support = batched_start_support(
        ready_models,
        rngs,
        paradigm.film_items,
        FILM_CUE_REINSTATEMENT,
    )
    recalls, cue_attempts, cue_items = batched_cued_recall(
        ready_models,
        rngs,
        paradigm.film_items,
        FILM_CUE_REINSTATEMENT,
        paradigm.max_recall,
        CUE_INTERVAL,
        FIRST_CUE_AFTER,
    )

    support_matrix = np.asarray(support).reshape(-1, n_presented)
    support_curve = np.mean(support_matrix, axis=0)
    support_denom = float(np.sum(support_curve))
    support_share = support_curve / support_denom if support_denom > 0 else np.full_like(support_curve, np.nan)

    raw_recalls = np.asarray(recalls).reshape(-1, recalls.shape[-1])
    remapped_recalls = np.asarray(
        standard_remap(
            raw_recalls,
            paradigm,
            show_break=True,
        )
    )
    pfr = np.asarray(
        fixed_pres_pnr(
            remapped_recalls,
            n_presented,
            query_recall_position=0,
        ),
        dtype=float,
    )
    spc = np.asarray(fixed_pres_spc(remapped_recalls, n_presented), dtype=float)
    totals = phase_recall_totals(spc)

    key = setting["label"]
    support_curves[key] = support_curve
    pfr_curves[key] = pfr
    spc_curves[key] = spc
    recall_sequences[key] = remapped_recalls

    for phase, total in totals.items():
        phase_rows.append({
            **common_metadata,
            "retrieval_setting": key,
            "start_drift_scale": start_drift_scale,
            "phase": phase,
            "recall_probability_mass": total,
        })

    for position, phase in enumerate(position_phase, start=1):
        support_rows.append({
            **common_metadata,
            "retrieval_setting": key,
            "start_drift_scale": start_drift_scale,
            "position": position,
            "phase": phase,
            "raw_support": float(support_curve[position - 1]),
            "support_share": float(support_share[position - 1]),
        })
        pfr_rows.append({
            **common_metadata,
            "retrieval_setting": key,
            "start_drift_scale": start_drift_scale,
            "position": position,
            "phase": phase,
            "first_recall_probability": float(pfr[position - 1]),
        })
        spc_rows.append({
            **common_metadata,
            "retrieval_setting": key,
            "start_drift_scale": start_drift_scale,
            "position": position,
            "phase": phase,
            "recall_probability": float(spc[position - 1]),
        })


In [ ]:
if data_dir is not None:
    write_rows(
        data_dir / f"{FIGURE_STR}_support.csv",
        [
            "retrieval_setting",
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "cue_interval",
            "film_source_start_drift_rate",
            "position",
            "phase",
            "raw_support",
            "support_share",
        ],
        support_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_pfr.csv",
        [
            "retrieval_setting",
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "cue_interval",
            "film_source_start_drift_rate",
            "position",
            "phase",
            "first_recall_probability",
        ],
        pfr_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_spc.csv",
        [
            "retrieval_setting",
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "cue_interval",
            "film_source_start_drift_rate",
            "position",
            "phase",
            "recall_probability",
        ],
        spc_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_phase_totals.csv",
        [
            "retrieval_setting",
            "start_drift_scale",
            "rejected_recall_drift_scale",
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "cue_interval",
            "film_source_start_drift_rate",
            "phase",
            "recall_probability_mass",
        ],
        phase_rows,
    )


In [ ]:
film_positions = [i for i, phase in enumerate(position_phase) if phase == "film"]
task_positions = [i for i, phase in enumerate(position_phase) if phase == "task"]
for setting in RETRIEVAL_SETTINGS:
    label = setting["label"]
    film_support = float(np.sum(np.asarray(support_curves[label])[film_positions]))
    task_support = float(np.sum(np.asarray(support_curves[label])[task_positions]))
    film_pfr = float(np.sum(np.asarray(pfr_curves[label])[film_positions]))
    task_pfr = float(np.sum(np.asarray(pfr_curves[label])[task_positions]))
    film_spc = float(np.sum(np.asarray(spc_curves[label])[film_positions]))
    task_spc = float(np.sum(np.asarray(spc_curves[label])[task_positions]))
    print(label)
    print(f"  Retrieval-start support, film/task: {film_support:.3f} / {task_support:.3f}")
    print(f"  First-recall probability, film/task: {film_pfr:.3f} / {task_pfr:.3f}")
    print(f"  Recall probability mass, film/task: {film_spc:.3f} / {task_spc:.3f}")
